In [16]:
#======= QLORA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import torch
torch.cuda.empty_cache()

In [1]:
!pip install --upgrade bitsandbytes

In [3]:
model_name = "mistralai/Mistral-7B-v0.1"

In [4]:
# Quantization config (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)


In [5]:
# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,  # 4-bit QLoRA config
    device_map="auto"                # automatically offloads to CPU if GPU memory is tight
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [7]:
# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [8]:
# Apply LoRA on quantized model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


In [20]:
# Dataset
dataset = load_dataset("imdb", split="train[:1%]")

# Ensure the tokenizer has a padding token
tokenizer.pad_token = tokenizer.eos_token  # use EOS as PAD because Mistral tokenizer doesn’t have a padding token by default

# Tokenization function
def tokenize_batch(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    # labels must match input_ids shape
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize_batch, batched=True)
dataset = dataset.remove_columns(["text"])

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

In [21]:
# Training args
training_args = TrainingArguments(
    output_dir="./lora-output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    fp16=True,
    optim="paged_adamw_32bit"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

Step,Training Loss
10,2.307818
20,2.109709
30,2.078567


TrainOutput(global_step=32, training_loss=2.155625194311142, metrics={'train_runtime': 355.3163, 'train_samples_per_second': 0.704, 'train_steps_per_second': 0.09, 'total_flos': 2733110722560000.0, 'train_loss': 2.155625194311142, 'epoch': 1.0})

In [22]:
# Save
model.save_pretrained("qlora-model")
tokenizer.save_pretrained("qlora-model")

('qlora-model/tokenizer_config.json', 'qlora-model/tokenizer.json')